# Shuffle Optimization: Partition Tuning, Bucket Joins, and Reducing I/O

**Module**: 10_performance_and_optimization  
**Notebook**: 04_shuffle_optimization.ipynb  
**Environment**: Apache Spark 4.2.1 · Standalone Cluster · PySpark · Python 3.13 · Java 21  

---

## Learning Objectives

- **What will I learn?**: Techniques to optimize shuffles: right-sizing shuffle partitions, bucketing to eliminate joins, and AQE partition coalescing.
- **Why does this topic exist?**: Shuffle optimization yields the highest ROI for big data jobs.
- **What problem does it solve?**: Default 200 shuffle partitions creating massive scheduling latency on small data, or spilling on large data.
- **Where is it used in real-world Data Engineering?**: Tuning heavy ETL jobs with multi-terabyte fact tables.

---

<!-- curriculum-enrichment:v1 -->

## Learning Contract

| Attribute | Value |
|---|---|
| **Priority** | Important |
| **Estimated effort** | 3–4 h |
| **Prerequisites** | Shuffle internals, partitioning, joins |
| **Required depth** | Working knowledge: can reduce shuffle volume and choose a shuffle layout |
| **Data Engineer** | Reduce shuffle bytes and tune partition counts and layouts. |
| **Senior Data Engineer** | Diagnose shuffle-heavy stages and choose between broadcast, bucketing, and pre-aggregation. |
| **Spark specialist** | Reason about serialization, compression, and shuffle service behavior. |
| **Next topic** | `10_performance_and_optimization/05_memory_optimization.ipynb` |

> **How to use this notebook**: read the problem first, then run the code, then connect the
> mechanism to the real-world cases and production notes. The goal is not to memorize API
> syntax; it is to be able to explain and defend a design decision.

---

## Problem and Motivation

### Why this concept exists

Shuffle optimization is mostly about moving fewer bytes: project columns, filter rows, pre-aggregate, and
avoid repeated exchanges on the same key. Partition count and layout determine how evenly the remaining bytes are
processed. A faster serializer or compression codec can help, but it cannot compensate for shuffling data that
should never have crossed the network.

### What happens if you ignore it

A team can still write code that runs on small data. The failure appears later: a job that
looks correct becomes slow, unstable, or expensive at scale, and the team lacks the vocabulary
to diagnose which Spark mechanism is responsible. This notebook builds that vocabulary and
ties it to observable behavior in the lab.

### A small mental model

Think of the topic as a contract between the logical work you describe and the physical work
Spark performs. The contract has a cost, a failure mode, and a set of trade-offs. The sections
below make each of those explicit.

---

## Core Concepts

### Definitions

| Term | Plain-language meaning | Technical meaning |
|---|---|---|
| **Shuffle write/read** | Bytes moved | Primary cost metric |
| **Partition count** | Parallelism after shuffle | Too few means large tasks; too many means overhead |
| **Broadcast** | Avoid shuffle | Small side replicated to executors |
| **Bucketing** | Pre-shuffle layout | Can eliminate repeated joins |
| **Compression** | Reduce network bytes | Codec trade-off between CPU and size |
| **Serializer** | Object-to-bytes | Kryo can be more compact than Java serialization |

### How it works, step by step

1. Identify the exchange in the plan.
2. Measure shuffle write and read in the UI.
3. Reduce the input with projection, filtering, or pre-aggregation.
4. Choose broadcast if one side is small.
5. Tune the partition count or rely on AQE coalescing.
6. Re-measure and check for skew.

### What to keep in mind

- The mechanism is not free; it consumes CPU, memory, network, or disk.
- The right choice depends on data size, distribution, latency target, and failure tolerance.
- A plan or UI observation is stronger evidence than an intuition about what "should" happen.

### 1. Initialize SparkSession on Standalone Cluster

We connect to `spark://spark-master:7077` with the driver configured to bind to container hostname `jupyter`.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
import time

spark = (
    SparkSession.builder
    .appName("Shuffle_Opt_Lab")
    .master("spark://spark-master:7077")
    .config("spark.driver.host", "jupyter")
    .config("spark.driver.bindAddress", "0.0.0.0")
    .config("spark.eventLog.enabled", "true")
    .config("spark.eventLog.dir", "file:///opt/spark-events")
    .config("spark.sql.warehouse.dir", "file:///opt/spark-data/warehouse")
    .config("spark.hadoop.fs.permissions.umask-mode", "000")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

print(f" SparkSession initialized successfully!")
print(f"Spark Version:      {spark.version}")
print(f"Master URL:         {spark.sparkContext.master}")
print(f"App Name:           {spark.sparkContext.appName}")
print(f"Default Parallelism: {spark.sparkContext.defaultParallelism}")
print(f"Spark UI available at: http://localhost:4040")

In [ ]:
# Tune shuffle partitions
spark.conf.set("spark.sql.shuffle.partitions", "8") # Match cluster capacity
orders = spark.read.option("header", True).csv("/opt/spark-data/retail/orders.csv")
res = orders.groupBy("channel").count()
res.show()
print("Shuffle executed with 8 partitions!")

---

## Experiment: AQE Automatic Partition Coalescing

- **1. What are we testing?**: Verify AQE coalesces 200 partitions into fewer partitions at runtime.
- **2. What do we expect?**: AQE merges small partitions automatically.

In [ ]:
spark.conf.set("spark.sql.shuffle.partitions", "100") # Deliberately oversized
res2 = orders.groupBy("status").count()
res2.show()
print("AQE coalesced partitions automatically in runtime plan!")
spark.conf.set("spark.sql.shuffle.partitions", "8")

**3. What happened?**: AQE reduced 100 shuffle partitions to fewer active post-shuffle tasks.  
**4. Why did it happen?**: spark.sql.adaptive.coalescePartitions.enabled was active.  
**5. Practical Engineering Lesson**: AQE protects against oversized shuffle partition settings.

---

## Real-World Use Cases

> The scenarios below are illustrative. Company names are omitted, and any numbers are hypothetical unless they are measured in this lab.

### Use Case 1: Repeated join on the same key

- **Industry**: Telecom
- **Scenario**: A pipeline joins the same two large tables in several steps.
- **Problem**: Each join shuffles the same data again.
- **Why this concept is relevant**: Bucketing both tables by the join key or materializing an intermediate join can remove repeated exchanges.
- **Implementation idea**: Compare the plan with and without bucketing; verify bucket compatibility.
- **Expected result**: Fewer shuffles and a shorter pipeline.
- **Alternative**: Cache the joined result.
- **Trade-offs**: A cache avoids recomputation but is ephemeral and may be evicted.

### Use Case 2: Wide-row shuffle

- **Industry**: Healthcare
- **Scenario**: A claims pipeline shuffles records with many unused fields.
- **Problem**: The shuffle writes far more bytes than the aggregation needs.
- **Why this concept is relevant**: Projecting only the necessary columns before the exchange reduces network and disk I/O.
- **Implementation idea**: Measure shuffle write before and after projection.
- **Expected result**: A smaller, faster shuffle.
- **Alternative**: Compress the shuffle more aggressively.
- **Trade-offs**: Compression reduces bytes but adds CPU cost and does not remove unused data.

---

## Business Value

- Lower network and disk cost.
- Faster joins and aggregations.
- More predictable stage durations.
- Better use of cluster capacity.

> **Note on numbers**: this repository does not claim production benchmarks. If you want to
> quantify a business case, measure the job in your own environment (runtime, shuffle bytes,
> spill, infrastructure cost) before and after the change.

---

## Production Engineering

### What changes when this runs in production

- Reduce data before the shuffle.
- Prefer broadcast for small sides.
- Use bucketing only for repeated large-large joins.
- Let AQE coalesce partitions when volume varies.
- Measure shuffle bytes, not just stage time.

### Investigation checklist

- [ ] Open the Spark UI and identify the relevant tab for this topic.
- [ ] Compare the metric before and after a configuration or code change.
- [ ] Check driver and executor logs for warnings or retries.
- [ ] Confirm that the fix is measured, not assumed.

### Common production mistakes

1. Copying a local-lab setting into a production cluster without sizing it.
2. Optimizing a single job without checking the downstream SLA and data-quality impact.
3. Treating a successful run as proof of correctness; verify row counts, keys, and invariants.
4. Forgetting that retries and backfills must be idempotent.

### Experiment: Cut shuffle bytes with projection

**Steps**

1. Run a join that selects a few columns but reads a wide DataFrame.
2. Record shuffle write.
3. Project before the join and record shuffle write again.

**What to observe**: Only the projected columns cross the shuffle boundary.

**Healthy result**: Shuffle bytes fall substantially.

**Problematic result**: The optimizer cannot push the projection because of a UDF or expression.

**Engineering takeaway**: The cheapest shuffle is the one with less data.

---

## Trade-offs and Alternatives

### Advantages

- Large performance gains
- Multiple techniques
- Supported by AQE

### Limitations

- Bucketing has strict rules
- Broadcast is memory-bound
- Compression adds CPU

### When to use it

- Join-heavy pipelines
- Aggregations
- Repeated ETL

### When not to use it

- Shuffling unused columns
- Forcing broadcast
- Tuning partitions blindly

### Alternatives inside Spark

- Broadcast
- Bucketing
- AQE
- Kryo

### Alternatives outside Spark

- Pre-joined tables
- Warehouse engines
- Data lake clustering

### Performance implications

Every choice has a cost. Measure runtime, shuffle read/write, spill, GC time, and task
duration percentiles before and after the change. A local four-core cluster is useful for
observing mechanics, but it is not a production benchmark.

---

## Practical Exercises

### Exercise 1 (Beginner): Set Shuffle Partitions
**Problem Statement**: Set shuffle partitions to 16.  
**Input / Context**: spark.conf

In [ ]:
# Write your solution for Exercise 1 here

### Exercise 2 (Intermediate): Bucketing for Zero Shuffle
**Problem Statement**: Explain bucketBy.  
**Input / Context**: Provided dataset

In [ ]:
# Write your solution for Exercise 2 here

### Exercise 3 (Advanced): Shuffle Compression Codec
**Problem Statement**: Check spark.io.compression.codec.  
**Input / Context**: Provided dataset

In [ ]:
# Write your solution for Exercise 3 here

---

## Production Debugging Exercise

### Bucketing did not remove the shuffle

**Scenario**: Both tables are bucketed, but the join plan still contains an exchange.

**Your task**: identify the most likely root cause, propose one diagnostic step, and propose
one fix. Write your answer before opening the model approach.

**Hints**

1. Check the bucket count and join key on both sides.
2. Confirm the tables are read through the catalog, not raw paths.
3. Verify the join key types match.

<details>
<summary>Model approach (open after trying)</summary>

1. Reproduce the issue with the smallest possible dataset.
2. Use the Spark UI and the physical plan to locate the stage or operator that dominates.
3. Change one variable at a time and re-measure.
4. Record the before/after metric so the fix is defensible.

The exact fix depends on the evidence. The important habit is to separate allocation,
scheduling, shuffle, memory, and data-quality causes instead of guessing.

</details>

---

## Reference Solutions

<details>
<summary><b>Click to expand full solutions</b></summary>

#### Solution 1 (Set Shuffle Partitions):
```python
spark.conf.set('spark.sql.shuffle.partitions','16')
```

#### Solution 2 (Bucketing for Zero Shuffle):
```python
print('Pre-partitions data by hash on disk; subsequent joins on that key require zero shuffle.')
```

#### Solution 3 (Shuffle Compression Codec):
```python
print(spark.conf.get('spark.io.compression.codec','lz4'))
```

</details>

---

## Module Summary & Interview Guide

### Key Takeaways
- **Sizing**: Target 100-200 MB per shuffle partition.
- **AQE**: Enables dynamic coalescing.

### Essential Technical Terms
- `shuffle.partitions`: Output partitions after wide transformation.
- `Bucketing`: Pre-sorted storage partitioning.

### Common Engineering Mistakes
- **Leaving default 200 on tiny data**: Creates 200 tiny tasks with massive scheduling overhead.

### Interview Questions on This Topic
- **Q: How does bucketing eliminate join shuffles?**
  - *A*: If two tables are bucketed on the same join key with the same number of buckets, corresponding buckets reside on the same partition index, allowing a narrow join without a shuffle.


### More Interview Questions

- **Q: How do you reduce shuffle volume?**
  - *A*: Project only needed columns, filter rows early, pre-aggregate, and avoid repeated exchanges on the same key.
- **Q: When does bucketing help?**
  - *A*: When both sides of a repeated join are bucketed by the same key with compatible bucket counts, allowing the planner to avoid a shuffle.

### What to Study Next
Open 05_memory_optimization.ipynb

---

### Tear Down SparkSession
Always close your session when finishing to release executor threads and driver ports.

In [ ]:
spark.stop()
print('SparkSession successfully stopped.')